# TradeView — ONNX Model Export Notebook

This notebook trains the **LSTM price forecasting model** and exports both models to ONNX format for use with TradeView on Render.

**Runtime:** GPU (T4) recommended — set via `Runtime → Change runtime type → T4 GPU`

### What this notebook does:
1. Installs dependencies
2. Downloads 5 years of NIFTY50 price data
3. Trains an LSTM model
4. Exports `lstm_price.onnx`
5. Exports `sentiment_distilbert.onnx`
6. Downloads all files to your computer

**Then:** Put the `.onnx` files in your project's `models/` folder and push to GitHub.

## Step 1 — Install Dependencies

In [ ]:
!pip install tf2onnx yfinance optimum[onnxruntime] onnx onnxruntime -q
print('✅ Dependencies installed')

## Step 2 — Configuration

In [ ]:
import os
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────
SEQUENCE_LENGTH = 60   # Number of past days the model looks at
FORECAST_DAYS  = 30   # Days ahead to forecast
EPOCHS         = 100  # Max training epochs (early stopping will kick in)
BATCH_SIZE     = 32

# Ticker to train on (NIFTY50 index)
TICKER = '^NSEI'
PERIOD = '5y'

os.makedirs('/content/models', exist_ok=True)
print(f'Training on: {TICKER} | Sequence: {SEQUENCE_LENGTH} days | Forecast: {FORECAST_DAYS} days')

## Step 3 — Download Training Data

In [ ]:
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

print(f'📥 Downloading {TICKER} data ({PERIOD})...')
ticker = yf.Ticker(TICKER)
df = ticker.history(period=PERIOD)

if df.empty:
    raise ValueError('Failed to fetch data. Check ticker symbol or internet connection.')

prices = df['Close'].values.reshape(-1, 1)
print(f'✅ Downloaded {len(prices)} trading days ({df.index[0].date()} → {df.index[-1].date()})')

# Scale
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(prices)
print(f'Price range: ₹{prices.min():.0f} – ₹{prices.max():.0f}')

# Preview
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 4))
plt.plot(df.index, prices, linewidth=1.5, color='#00d4ff')
plt.title(f'{TICKER} — Closing Price (Training Data)', fontsize=14)
plt.xlabel('Date'); plt.ylabel('Price (₹)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Step 4 — Build Sequences & Split Data

In [ ]:
X, y = [], []
for i in range(len(scaled) - SEQUENCE_LENGTH - FORECAST_DAYS + 1):
    X.append(scaled[i:i + SEQUENCE_LENGTH])
    y.append(scaled[i + SEQUENCE_LENGTH])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f'Total sequences : {len(X)}')
print(f'Training set    : {len(X_train)} sequences')
print(f'Validation set  : {len(X_val)} sequences')
print(f'Input shape     : {X.shape}')

## Step 5 — Build & Train LSTM Model

In [ ]:
import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, 1)),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1),
], name='TradeView_LSTM')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

print('🚀 Training started...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print('✅ Training complete!')

## Step 6 — Validate & Plot Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Predict on validation set
val_preds_scaled = model.predict(X_val, verbose=0)
val_preds = scaler.inverse_transform(val_preds_scaled)
val_actual = scaler.inverse_transform(y_val)

axes[1].plot(val_actual, label='Actual', linewidth=1.5)
axes[1].plot(val_preds, label='Predicted', linewidth=1.5, linestyle='--')
axes[1].set_title('Validation: Actual vs Predicted'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Metrics
mae = np.mean(np.abs(val_preds - val_actual))
rmse = np.sqrt(np.mean((val_preds - val_actual)**2))
dir_acc = np.mean(np.sign(np.diff(val_preds.flatten())) == np.sign(np.diff(val_actual.flatten())))
print(f'MAE:                  ₹{mae:.2f}')
print(f'RMSE:                 ₹{rmse:.2f}')
print(f'Directional Accuracy: {dir_acc:.1%}')

## Step 7 — Export LSTM to ONNX

In [ ]:
import tf2onnx

output_path = '/content/models/lstm_price.onnx'
input_signature = [
    tf.TensorSpec(shape=(None, SEQUENCE_LENGTH, 1), dtype=tf.float32, name='input')
]

print('🔄 Converting model to ONNX...')
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=13,
    output_path=output_path
)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'✅ Saved: lstm_price.onnx ({size_mb:.1f} MB)')

In [ ]:
# Quick ONNX verification
import onnxruntime as ort

session = ort.InferenceSession(output_path)
input_name = session.get_inputs()[0].name
test_input = scaled[-SEQUENCE_LENGTH:].reshape(1, SEQUENCE_LENGTH, 1).astype(np.float32)
onnx_pred = session.run(None, {input_name: test_input})[0]
keras_pred = model.predict(test_input, verbose=0)

print(f'Keras output : {scaler.inverse_transform(keras_pred)[0,0]:.2f}')
print(f'ONNX output  : {scaler.inverse_transform(onnx_pred)[0,0]:.2f}')
print(f'✅ ONNX output matches Keras — model verified!')

## Step 8 — Export DistilBERT Sentiment Model to ONNX

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
EXPORT_DIR = '/content/models/sentiment_export'

print(f'📥 Downloading and exporting {MODEL_NAME} to ONNX...')
ort_model = ORTModelForSequenceClassification.from_pretrained(MODEL_NAME, export=True)
ort_model.save_pretrained(EXPORT_DIR)

import shutil, glob
onnx_files = glob.glob(f'{EXPORT_DIR}/*.onnx')
if onnx_files:
    shutil.copy(onnx_files[0], '/content/models/sentiment_distilbert.onnx')
    size_mb = os.path.getsize('/content/models/sentiment_distilbert.onnx') / 1024 / 1024
    print(f'✅ Saved: sentiment_distilbert.onnx ({size_mb:.1f} MB)')
else:
    print('❌ ONNX file not found in export dir')

# Save tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained('/content/models')
print('✅ Tokenizer saved to models/')

In [ ]:
# Quick sentiment ONNX verification
from tokenizers import Tokenizer
import json

sent_session = ort.InferenceSession('/content/models/sentiment_distilbert.onnx')
test_tokenizer = AutoTokenizer.from_pretrained('/content/models')

test_texts = [
    'Markets rally strongly on positive earnings',
    'Stock market crashes on recession fears'
]
for text in test_texts:
    enc = test_tokenizer(text, return_tensors='np', truncation=True, max_length=128, padding='max_length')
    inputs = {'input_ids': enc['input_ids'].astype(np.int64), 'attention_mask': enc['attention_mask'].astype(np.int64)}
    logits = sent_session.run(None, inputs)[0]
    label = ['NEGATIVE', 'POSITIVE'][int(np.argmax(logits))]
    print(f'  "{text[:40]}..." → {label}')

print('✅ Sentiment ONNX model verified!')

## Step 9 — Download Files to Your Computer

In [ ]:
import os

# List all generated files
print('Files in /content/models:')
for f in sorted(os.listdir('/content/models')):
    path = f'/content/models/{f}'
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1024
        print(f'  {f:<45} {size:>8.1f} KB')

In [ ]:
from google.colab import files

print('⬇️  Downloading lstm_price.onnx...')
files.download('/content/models/lstm_price.onnx')

print('⬇️  Downloading sentiment_distilbert.onnx...')
files.download('/content/models/sentiment_distilbert.onnx')

print()
print('✅ All done!')
print()
print('Next steps:')
print('  1. Move the downloaded .onnx files into your project: Trade_View/models/')
print('  2. git add models/lstm_price.onnx models/sentiment_distilbert.onnx')
print('  3. git commit -m "Add pre-trained ONNX models"')
print('  4. git push  →  Render auto-deploys with full ONNX inference!')